# 🎬 낙상 감지 데이터셋 전처리 파이프라인

영상 데이터를 학습에 적합한 형태로 전처리합니다.

### 📋 전처리 과정
1. **사람 감지**: YOLO로 첫 프레임에서 사람 위치 파악
2. **1:1 크롭**: 사람 중심으로 정사각형 영역 추출
3. **리사이즈**: 256×256 해상도로 축소
4. **MP4 출력**: GPU 가속 인코딩으로 빠른 처리

### ⚡ 성능 최적화
- **FFmpeg NVENC**: GPU 하드웨어 인코딩 (A100 지원)
- **병렬 처리**: 여러 영상 동시 처리
- **로컬 캐시**: Drive I/O 병목 해소

---

In [1]:
# @title 📁 환경 설정 및 경로 지정
# @markdown Google Drive를 마운트하고 데이터셋 경로를 설정합니다.

import os
from google.colab import drive

# @markdown ---
# @markdown ### 📂 경로 설정
src_path = 'drive/MyDrive/졸업 과제/dataset/source'  # @param {type:"string"}
# @markdown > `src_path`: 원본 영상이 있는 폴더

dst_path = 'drive/MyDrive/졸업 과제/dataset/preprocessed'  # @param {type:"string"}
# @markdown > `dst_path`: 전처리된 영상을 저장할 폴더

metadata_path = 'drive/MyDrive/졸업 과제/dataset/metadata'  # @param {type:"string"}
# @markdown > `metadata_path`: SQLite DB 저장 위치

# Drive 마운트
drive.mount('/content/drive')

# 디렉토리 생성
os.makedirs(dst_path, exist_ok=True)
os.makedirs(metadata_path, exist_ok=True)

print("✅ 경로 설정 완료")
print(f"  📥 소스: {src_path}")
print(f"  📤 출력: {dst_path}")
print(f"  🗄️ 메타: {metadata_path}")




Mounted at /content/drive
✅ 경로 설정 완료
  📥 소스: drive/MyDrive/졸업 과제/dataset/source
  📤 출력: drive/MyDrive/졸업 과제/dataset/preprocessed
  🗄️ 메타: drive/MyDrive/졸업 과제/dataset/metadata


In [2]:
# @title 📦 필수 라이브러리 설치
# @markdown 전처리에 필요한 Python 패키지를 설치합니다.

import sys
import subprocess

def install_package(package, display_name=None):
    """패키지 설치 및 결과 출력"""
    if display_name is None:
        display_name = package

    print(f"📦 {display_name} 설치 중...", end=" ")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        print("✅")
    except subprocess.CalledProcessError:
        print("❌ (설치 실패)")

print("=" * 60)
print("🔧 의존성 설치 시작")
print("=" * 60)

# 필수 패키지 설치
install_package("ultralytics", "Ultralytics YOLO")
install_package("opencv-python-headless", "OpenCV")
install_package("tqdm", "tqdm (진행바)")

print("=" * 60)
print("✅ 설치 완료")
print("=" * 60)

# 설치된 버전 확인
print("\n📋 설치된 버전:")
try:
    import ultralytics
    print(f"  • Ultralytics: {ultralytics.__version__}")
except:
    pass

try:
    import cv2
    print(f"  • OpenCV: {cv2.__version__}")
except:
    pass

try:
    import torch
    print(f"  • PyTorch: {torch.__version__}")
    print(f"  • CUDA: {'사용 가능' if torch.cuda.is_available() else '사용 불가'}")
except:
    pass

🔧 의존성 설치 시작
📦 Ultralytics YOLO 설치 중... ✅
📦 OpenCV 설치 중... ✅
📦 tqdm (진행바) 설치 중... ✅
✅ 설치 완료

📋 설치된 버전:
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  • Ultralytics: 8.4.7
  • OpenCV: 4.12.0
  • PyTorch: 2.9.0+cu126
  • CUDA: 사용 가능


## 🗄️ Step 1.5. 영상 목록 데이터베이스 초기화

- SQLite를 사용하여 처리 상태 관리
- 세션이 끊겨도 이어서 처리 가능
- 상태: `pending` → `processing` → `completed` / `error`

In [9]:
# @title 🗄️ 영상 목록 데이터베이스 초기화 (최적화 + 백업)
# @markdown 소스 폴더를 스캔하여 영상 목록을 SQLite DB에 저장합니다.

import sqlite3
import os
import shutil
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

# @markdown ---
# @markdown ### ⚙️ 설정
need_init = False  # @param {type:"boolean"}
# @markdown > `need_init`: True면 DB 초기화 및 영상 스캔 실행

use_backup = True  # @param {type:"boolean"}
# @markdown > `use_backup`: True면 백업 사용 (2번째 실행부터 빠름)

db_path = os.path.join(metadata_path, 'manager.db')
backup_path = os.path.join(metadata_path, 'manager.db.backup')

def scan_folder_fast(label_folder, base_path):
    """
    단일 폴더를 빠르게 스캔 (os.scandir 사용)

    os.scandir는 os.walk와 glob보다 빠름:
    - stat 정보를 캐시하여 추가 시스템 콜 불필요
    - 디렉토리 순회 최적화
    """
    video_files = []
    label_path = os.path.join(base_path, label_folder)

    if not os.path.exists(label_path):
        return []

    # os.scandir로 재귀적 스캔 (가장 빠름)
    def scan_recursive(path):
        try:
            with os.scandir(path) as entries:
                for entry in entries:
                    if entry.is_file() and entry.name.endswith('.mp4'):
                        video_files.append((entry.path, label_folder))
                    elif entry.is_dir():
                        scan_recursive(entry.path)
        except PermissionError:
            pass  # 권한 없는 폴더는 건너뛰기

    scan_recursive(label_path)
    return video_files

def initialize_and_fill_db():
    """DB 테이블 생성 및 영상 목록 등록 (병렬 스캔 + 백업)"""

    # 백업 복원 시도
    if use_backup and os.path.exists(backup_path) and not os.path.exists(db_path):
        print(f"💾 백업 발견! 복원 중...")
        shutil.copy2(backup_path, db_path)
        print(f"✅ 백업 복원 완료 (스캔 건너뜀)")
        return

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # 테이블 생성
    cur.execute('''
        CREATE TABLE IF NOT EXISTS videos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            video_id TEXT UNIQUE,
            video_path TEXT UNIQUE,
            status TEXT DEFAULT 'pending',
            frame_count INTEGER DEFAULT 0,
            label TEXT,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    conn.commit()

    print(f"🔍 스캔 시작: {src_path}")
    print(f"  대상: Y/**, N/** 폴더")
    print(f"  방식: 병렬 스캔 (os.scandir)")

    # =========================================================================
    # 병렬로 Y, N 폴더 스캔 (훨씬 빠름)
    # =========================================================================
    scan_start = datetime.now()

    with ThreadPoolExecutor(max_workers=2) as executor:
        # Y, N 폴더를 동시에 스캔
        futures = {
            executor.submit(scan_folder_fast, label, src_path): label
            for label in ['Y', 'N']
        }

        all_videos = []
        for future in futures:
            label = futures[future]
            videos = future.result()
            print(f"  📂 {label}/ 폴더: {len(videos)}개 발견")
            all_videos.extend(videos)

    scan_time = (datetime.now() - scan_start).total_seconds()
    print(f"  ⏱️ 스캔 완료: {scan_time:.1f}초")

    # =========================================================================
    # DB에 일괄 삽입 (배치 처리로 더 빠름)
    # =========================================================================
    print(f"\n💾 DB에 저장 중...")
    insert_start = datetime.now()

    new_count = 0
    batch = []
    BATCH_SIZE = 1000  # 1000개씩 배치 삽입

    for abs_path, label in tqdm(all_videos, desc="  저장"):
        rel_path = os.path.relpath(abs_path, src_path)
        video_id = os.path.basename(os.path.dirname(rel_path))

        batch.append((video_id, rel_path, label))

        # 배치가 차면 일괄 삽입
        if len(batch) >= BATCH_SIZE:
            try:
                cur.executemany(
                    "INSERT OR IGNORE INTO videos (video_id, video_path, label) VALUES (?, ?, ?)",
                    batch
                )
                new_count += cur.rowcount
                conn.commit()
                batch = []
            except sqlite3.Error as e:
                print(f"    ❌ 배치 삽입 에러: {e}")
                batch = []

    # 남은 배치 처리
    if batch:
        try:
            cur.executemany(
                "INSERT OR IGNORE INTO videos (video_id, video_path, label) VALUES (?, ?, ?)",
                batch
            )
            new_count += cur.rowcount
            conn.commit()
        except sqlite3.Error as e:
            print(f"    ❌ 배치 삽입 에러: {e}")

    conn.close()

    insert_time = (datetime.now() - insert_start).total_seconds()
    print(f"  ⏱️ 저장 완료: {insert_time:.1f}초")

    total_time = scan_time + insert_time
    print(f"\n✅ 완료: {new_count}개 등록 (총 {total_time:.1f}초)")

    # =========================================================================
    # DB 백업 생성
    # =========================================================================
    if use_backup and new_count > 0:
        print(f"\n💾 백업 생성 중...")
        try:
            shutil.copy2(db_path, backup_path)
            backup_size = os.path.getsize(backup_path) / 1024  # KB
            print(f"✅ 백업 완료: {backup_path}")
            print(f"  크기: {backup_size:.1f} KB")
            print(f"  💡 다음 실행시 백업 사용 (스캔 불필요)")
        except Exception as e:
            print(f"⚠️ 백업 실패: {e}")

def check_db_status():
    """현재 DB 상태 출력"""
    if not os.path.exists(db_path):
        print("⚠️ DB 파일이 없습니다. need_init=True로 실행하세요.")
        return

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # 상태별 카운트
    cur.execute("SELECT status, COUNT(*) FROM videos GROUP BY status")
    status_results = cur.fetchall()

    print("\n📊 현재 DB 상태:")
    total = 0
    for status, count in status_results:
        emoji = {'pending': '⏳', 'processing': '🔄', 'completed': '✅', 'error': '❌'}.get(status, '❓')
        print(f"  {emoji} {status}: {count}개")
        total += count

    # 라벨별 카운트
    cur.execute("SELECT label, COUNT(*) FROM videos GROUP BY label")
    label_results = cur.fetchall()

    if label_results:
        print(f"\n  라벨별 분포:")
        for label, count in label_results:
            if label:
                print(f"    {label}: {count}개")

    print(f"\n  📁 총 영상: {total}개")

    # DB 파일 크기
    db_size = os.path.getsize(db_path) / 1024  # KB
    print(f"  💾 DB 크기: {db_size:.1f} KB")

    # 백업 상태
    if os.path.exists(backup_path):
        backup_time = datetime.fromtimestamp(os.path.getmtime(backup_path))
        print(f"  📦 백업 있음: {backup_time.strftime('%Y-%m-%d %H:%M:%S')}")

    conn.close()

# 실행
if need_init:
    # DB가 있고 백업도 있으면 사용자 확인
    if os.path.exists(db_path) and use_backup:
        print("⚠️ DB가 이미 존재합니다.")
        print("  need_init=True는 DB를 다시 생성합니다.")
        print("  기존 DB를 유지하려면 need_init=False로 변경하세요.\n")

    initialize_and_fill_db()
elif not os.path.exists(db_path) and os.path.exists(backup_path) and use_backup:
    # DB는 없지만 백업이 있으면 자동 복원
    print(f"💾 백업 발견! 자동 복원 중...")
    shutil.copy2(backup_path, db_path)
    print(f"✅ 백업 복원 완료")

check_db_status()


📊 현재 DB 상태:
  ⏳ pending: 20400개

  라벨별 분포:
    N: 5112개
    Y: 15288개

  📁 총 영상: 20400개
  💾 DB 크기: 3488.0 KB
  📦 백업 있음: 2026-01-27 08:46:15


## 🎯 Step 2. YOLO 모델 로드 및 유틸리티 함수

- **GPU 확인**: CUDA/GPU 정보 출력
- **YOLO 모델**: 사람 감지용 (첫 프레임에서 1회만 추론)
- **크롭 함수**: 중심점 기준 안전한 1:1 크롭 영역 계산

In [10]:
# @title 🎯 YOLO 모델 로드 및 GPU 확인
# @markdown 사람 감지용 YOLO 모델을 로드하고 GPU 상태를 확인합니다.

import cv2
import numpy as np
import os
import torch
from ultralytics import YOLO

# =============================================================================
# GPU 상태 확인
# =============================================================================
print("=" * 50)
print("🖥️ 시스템 정보")
print("=" * 50)
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA 사용 가능: {'✅ Yes' if torch.cuda.is_available() else '❌ No'}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  GPU: {gpu_name}")
    print(f"  GPU 메모리: {gpu_mem:.1f} GB")
else:
    print("  ⚠️ GPU 미사용 - CPU 모드로 실행됩니다")
print("=" * 50)

# =============================================================================
# YOLO 모델 로드
# =============================================================================
yolo_model = YOLO('yolov8n.pt')
print(f"\n🤖 YOLO 모델 로드 완료")
print(f"  디바이스: {yolo_model.device}")

# =============================================================================
# 크롭 영역 계산 함수
# =============================================================================
def calculate_safe_crop_region(cx, cy, w, h):
    """
    중심점 기준 1:1 크롭 영역 계산

    Args:
        cx, cy: 중심점 좌표 (YOLO 감지 결과 또는 이미지 중앙)
        w, h: 원본 이미지 크기

    Returns:
        x1, y1: 크롭 시작 좌표
        crop_size: 크롭 영역 크기 (정사각형)

    Note:
        중심점이 가장자리에 있어도 크롭 영역이 이미지 밖으로
        벗어나지 않도록 자동 보정합니다.
    """
    crop_size = min(h, w)
    half_size = crop_size // 2

    # 중심점 보정 (이미지 경계 체크)
    safe_cx = cx
    safe_cy = cy

    if cx - half_size < 0:
        safe_cx = half_size
    elif cx + half_size > w:
        safe_cx = w - half_size

    if cy - half_size < 0:
        safe_cy = half_size
    elif cy + half_size > h:
        safe_cy = h - half_size

    # 최종 크롭 좌표
    x1 = max(0, min(safe_cx - half_size, w - crop_size))
    y1 = max(0, min(safe_cy - half_size, h - crop_size))

    return x1, y1, crop_size

print("\n✅ 유틸리티 함수 정의 완료")

🖥️ 시스템 정보
  PyTorch: 2.9.0+cu126
  CUDA 사용 가능: ✅ Yes
  GPU: Tesla T4
  GPU 메모리: 14.7 GB

🤖 YOLO 모델 로드 완료
  디바이스: cpu

✅ 유틸리티 함수 정의 완료


## 🚀 Step 3. 전체 영상 배치 처리 실행

### 처리 파이프라인
```
원본 영상 (Drive)
    ↓ 로컬 복사
첫 프레임 추출
    ↓ YOLO 추론 (GPU)
사람 중심점 감지
    ↓
크롭 영역 계산 (1:1)
    ↓ FFmpeg + NVENC (GPU)
256×256 MP4 출력
    ↓
결과 영상 (Drive)
```

### 설정값
| 변수 | 기본값 | 설명 |
|------|--------|------|
| `OUTPUT_SIZE` | 256 | 출력 해상도 |
| `NUM_WORKERS` | 4 | 동시 처리 영상 수 |
| `BATCH_SIZE` | 10 | 진행상황 출력 주기 |

In [11]:
# @title 🚀 전체 영상 배치 처리 (GPU 가속 + 병렬 처리)

# @markdown ### 📐 출력 설정
OUTPUT_SIZE = 256  # @param {type:"slider", min:64, max:512, step:64}
OUTPUT_FPS = 0  # @param {type:"integer"}
# @markdown > `OUTPUT_FPS`: 0이면 원본 FPS 유지, 양수면 해당 FPS로 변환

# @markdown ---
# @markdown ### ⚡ 성능 설정
NUM_WORKERS = 8  # @param {type:"slider", min:1, max:8, step:1}
# @markdown > `NUM_WORKERS`: 동시 처리 영상 수 (GPU 메모리에 따라 조절)

USE_NVENC = True  # @param {type:"boolean"}
# @markdown > `USE_NVENC`: GPU 인코딩 사용 (A100/V100/T4 지원)

# @markdown ---
# @markdown ### 📊 모니터링 설정
BATCH_SIZE = 10  # @param {type:"slider", min:5, max:100, step:5}
# @markdown > `BATCH_SIZE`: n개 처리마다 진행상황 출력 (세션 유지용)

import cv2
import numpy as np
import os
import shutil
import time
import sqlite3
import subprocess
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# =============================================================================
# 임시 작업 디렉토리 설정
# =============================================================================
LOCAL_WORK_DIR = '/content/temp_video'  # 입력 영상 임시 저장
LOCAL_OUT_DIR = '/content/temp_out'     # 출력 영상 임시 저장
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_OUT_DIR, exist_ok=True)

# =============================================================================
# NVENC (GPU 인코딩) 사용 가능 여부 확인
# =============================================================================
def check_nvenc_available():
    """FFmpeg에서 h264_nvenc 인코더 지원 여부 확인"""
    try:
        result = subprocess.run(
            ['ffmpeg', '-encoders'],
            capture_output=True, text=True, timeout=10
        )
        return 'h264_nvenc' in result.stdout
    except:
        return False

NVENC_AVAILABLE = check_nvenc_available() and USE_NVENC

# 설정 출력
print("=" * 60)
print("📋 전처리 설정")
print("=" * 60)
print(f"  출력 해상도: {OUTPUT_SIZE}x{OUTPUT_SIZE}")
print(f"  출력 FPS: {'원본 유지' if OUTPUT_FPS == 0 else f'{OUTPUT_FPS} fps'}")
print(f"  병렬 워커: {NUM_WORKERS}개")
print(f"  GPU 인코딩 (NVENC): {'✅ 사용' if NVENC_AVAILABLE else '❌ 미사용 (CPU 인코딩)'}")
print(f"  진행상황 출력: {BATCH_SIZE}개마다")
print("=" * 60)

# YOLO 모델 동시 접근 방지용 락
yolo_lock = threading.Lock()


# =============================================================================
# 단일 영상 처리 함수
# =============================================================================
def process_single_video_fast(video_rel_path, video_id, output_size, output_fps):
    """
    단일 영상을 전처리하여 MP4로 변환

    Args:
        video_rel_path: 원본 영상 상대 경로
        video_id: 영상 식별자
        output_size: 출력 해상도 (정사각형)
        output_fps: 출력 FPS (0이면 원본 유지)

    Returns:
        (성공여부, 프레임수, 메시지)
    """

    # -------------------------------------------------------------------------
    # 경로 설정
    # -------------------------------------------------------------------------
    src_file = os.path.join(src_path, video_rel_path)
    local_video_path = os.path.join(LOCAL_WORK_DIR, f"{video_id}_input.mp4")

    rel_dir = os.path.dirname(video_rel_path)
    final_dst_dir = os.path.join(dst_path, rel_dir)
    local_output_path = os.path.join(LOCAL_OUT_DIR, f"{video_id}.mp4")
    final_output_path = os.path.join(final_dst_dir, f"{video_id}.mp4")

    os.makedirs(final_dst_dir, exist_ok=True)

    try:
        # ---------------------------------------------------------------------
        # Step 1: Drive → 로컬 복사
        # ---------------------------------------------------------------------
        shutil.copy2(src_file, local_video_path)

        # ---------------------------------------------------------------------
        # Step 2: 영상 메타데이터 + 첫 프레임 추출
        # ---------------------------------------------------------------------
        cap = cv2.VideoCapture(local_video_path)
        if not cap.isOpened():
            return False, 0, "Video Open Error"

        original_fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        ret, first_frame = cap.read()
        cap.release()

        if not ret:
            return False, 0, "First Frame Read Error"

        # ---------------------------------------------------------------------
        # Step 3: YOLO로 사람 위치 감지 (GPU)
        # ---------------------------------------------------------------------
        cx, cy = w // 2, h // 2
        with yolo_lock:
            results = yolo_model(first_frame, verbose=False, classes=[0], conf=0.3)

        if len(results[0].boxes) > 0:
            box = results[0].boxes[0].xywh[0].cpu().numpy()
            cx, cy = int(box[0]), int(box[1])

        # ---------------------------------------------------------------------
        # Step 4: 크롭 영역 계산
        # ---------------------------------------------------------------------
        x1, y1, crop_size = calculate_safe_crop_region(cx, cy, w, h)

        # ---------------------------------------------------------------------
        # Step 5: FFmpeg 명령어 구성
        # ---------------------------------------------------------------------
        # 비디오 필터 구성
        vf_filters = f'crop={crop_size}:{crop_size}:{x1}:{y1},scale={output_size}:{output_size}'

        # FPS 변환 필터 추가 (output_fps > 0인 경우)
        if output_fps > 0:
            vf_filters += f',fps={output_fps}'

        if NVENC_AVAILABLE:
            # GPU 인코딩
            ffmpeg_cmd = [
                'ffmpeg', '-y',
                '-hwaccel', 'cuda',
                '-i', local_video_path,
                '-vf', vf_filters,
                '-c:v', 'h264_nvenc',
                '-preset', 'p4',
                '-an',
                local_output_path
            ]
        else:
            # CPU 인코딩
            ffmpeg_cmd = [
                'ffmpeg', '-y',
                '-i', local_video_path,
                '-vf', vf_filters,
                '-c:v', 'libx264',
                '-preset', 'ultrafast',
                '-an',
                local_output_path
            ]

        result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True, timeout=300)

        if result.returncode != 0:
            return False, 0, f"FFmpeg Error: {result.stderr[-200:]}"

        # ---------------------------------------------------------------------
        # Step 6: 결과 이동 + 정리
        # ---------------------------------------------------------------------
        if os.path.exists(local_output_path):
            shutil.move(local_output_path, final_output_path)

        if os.path.exists(local_video_path):
            os.remove(local_video_path)

        return True, total_frames, "Success"

    except Exception as e:
        for f in [local_video_path, local_output_path]:
            if os.path.exists(f):
                os.remove(f)
        return False, 0, str(e)


# =============================================================================
# 병렬 배치 처리 메인 함수
# =============================================================================
def run_batch_processing_parallel():
    """전체 영상을 병렬로 배치 처리"""

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute("SELECT COUNT(*) FROM videos WHERE status = 'pending'")
    total_pending = cur.fetchone()[0]

    if total_pending == 0:
        print("✅ 처리할 영상이 없습니다.")
        conn.close()
        return

    # -------------------------------------------------------------------------
    # 처리 시작 로그
    # -------------------------------------------------------------------------
    print(f"\n{'=' * 60}")
    print(f"🎬 [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 배치 처리 시작")
    print(f"{'=' * 60}")
    print(f"  대기 중: {total_pending}개")
    print(f"  출력: {OUTPUT_SIZE}x{OUTPUT_SIZE} MP4")
    print(f"  FPS: {'원본 유지' if OUTPUT_FPS == 0 else f'{OUTPUT_FPS}'}")
    print(f"  워커: {NUM_WORKERS}개 | GPU: {NVENC_AVAILABLE}")
    print(f"{'=' * 60}\n")

    processed_total = 0
    success_count = 0
    error_count = 0
    start_time = time.time()

    db_lock = threading.Lock()

    # -------------------------------------------------------------------------
    # DB 헬퍼 함수
    # -------------------------------------------------------------------------
    def get_next_video():
        with db_lock:
            conn_local = sqlite3.connect(db_path)
            cur_local = conn_local.cursor()
            cur_local.execute("SELECT video_id, video_path FROM videos WHERE status = 'pending' LIMIT 1")
            row = cur_local.fetchone()
            if row:
                cur_local.execute(
                    "UPDATE videos SET status = 'processing', updated_at = CURRENT_TIMESTAMP WHERE video_id = ?",
                    (row[0],)
                )
                conn_local.commit()
            conn_local.close()
            return row

    def update_video_status(video_id, status, frame_count=0):
        with db_lock:
            conn_local = sqlite3.connect(db_path)
            cur_local = conn_local.cursor()
            cur_local.execute(
                "UPDATE videos SET status = ?, frame_count = ?, updated_at = CURRENT_TIMESTAMP WHERE video_id = ?",
                (status, frame_count, video_id)
            )
            conn_local.commit()
            conn_local.close()

    def worker_task(video_id, video_path):
        return video_id, *process_single_video_fast(video_path, video_id, OUTPUT_SIZE, OUTPUT_FPS)

    # -------------------------------------------------------------------------
    # 병렬 처리 실행
    # -------------------------------------------------------------------------
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {}

        for _ in range(NUM_WORKERS):
            row = get_next_video()
            if row:
                video_id, video_path = row
                future = executor.submit(worker_task, video_id, video_path)
                futures[future] = video_id

        while futures:
            for future in as_completed(futures):
                video_id = futures.pop(future)

                try:
                    vid, success, frame_count, msg = future.result()

                    if success:
                        update_video_status(vid, 'completed', frame_count)
                        success_count += 1
                    else:
                        update_video_status(vid, 'error', 0)
                        error_count += 1
                        print(f"  ❌ [ERROR] {vid}: {msg}")

                    processed_total += 1

                    if processed_total % BATCH_SIZE == 0:
                        elapsed = time.time() - start_time
                        avg_time = elapsed / processed_total
                        remaining = total_pending - processed_total
                        eta = avg_time * remaining
                        speed = processed_total / (elapsed / 60)

                        print(f"📊 [{datetime.now().strftime('%H:%M:%S')}] "
                              f"{processed_total}/{total_pending} ({100*processed_total/total_pending:.1f}%) | "
                              f"✅{success_count} ❌{error_count} | "
                              f"{speed:.1f}개/분 | ETA: {eta/60:.1f}분")

                except Exception as e:
                    update_video_status(video_id, 'error', 0)
                    error_count += 1
                    processed_total += 1
                    print(f"  ❌ [ERROR] {video_id}: {str(e)}")

                row = get_next_video()
                if row:
                    new_video_id, new_video_path = row
                    new_future = executor.submit(worker_task, new_video_id, new_video_path)
                    futures[new_future] = new_video_id

                break

    conn.close()

    # -------------------------------------------------------------------------
    # 최종 결과
    # -------------------------------------------------------------------------
    total_time = time.time() - start_time
    print(f"\n{'=' * 60}")
    print(f"🏁 [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 배치 처리 완료")
    print(f"{'=' * 60}")
    print(f"  총 처리: {processed_total}개")
    print(f"  성공: {success_count}개 ✅")
    print(f"  실패: {error_count}개 ❌")
    print(f"  소요시간: {total_time/60:.1f}분 ({total_time/3600:.2f}시간)")
    if processed_total > 0:
        print(f"  평균: {total_time/processed_total:.2f}초/영상")
        print(f"  속도: {processed_total/(total_time/60):.1f}개/분")
    print(f"{'=' * 60}")


# =============================================================================
# 실행
# =============================================================================
run_batch_processing_parallel()

📋 전처리 설정
  출력 해상도: 256x256
  출력 FPS: 원본 유지
  병렬 워커: 8개
  GPU 인코딩 (NVENC): ✅ 사용
  진행상황 출력: 10개마다

🎬 [2026-01-27 08:58:02] 배치 처리 시작
  대기 중: 20400개
  출력: 256x256 MP4
  FPS: 원본 유지
  워커: 8개 | GPU: True

📊 [08:59:25] 10/20400 (0.0%) | ✅10 ❌0 | 7.2개/분 | ETA: 2823.7분
📊 [09:00:09] 20/20400 (0.1%) | ✅20 ❌0 | 9.4개/분 | ETA: 2162.1분
📊 [09:00:55] 30/20400 (0.1%) | ✅30 ❌0 | 10.4개/분 | ETA: 1967.2분
📊 [09:01:41] 40/20400 (0.2%) | ✅40 ❌0 | 11.0개/분 | ETA: 1859.2분
📊 [09:02:52] 50/20400 (0.2%) | ✅50 ❌0 | 10.3개/분 | ETA: 1968.7분
📊 [09:03:38] 60/20400 (0.3%) | ✅60 ❌0 | 10.7개/분 | ETA: 1901.2분
📊 [09:04:24] 70/20400 (0.3%) | ✅70 ❌0 | 11.0개/분 | ETA: 1851.6분
📊 [09:05:13] 80/20400 (0.4%) | ✅80 ❌0 | 11.1개/분 | ETA: 1824.3분
📊 [09:06:22] 90/20400 (0.4%) | ✅90 ❌0 | 10.8개/분 | ETA: 1883.2분
📊 [09:07:07] 100/20400 (0.5%) | ✅100 ❌0 | 11.0개/분 | ETA: 1844.2분
📊 [09:07:55] 110/20400 (0.5%) | ✅110 ❌0 | 11.1개/분 | ETA: 1825.2분
📊 [09:08:45] 120/20400 (0.6%) | ✅120 ❌0 | 11.2개/분 | ETA: 1812.6분
📊 [09:09:48] 130/20400 (0.6%) | ✅130 ❌0 | 

KeyboardInterrupt: 